# Retail Data Wrangling and Analytics

## Load SQL into Dataframe

In [0]:
%python
retail_df = spark.sql("SELECT * FROM `jrvs_data_analytics_workspace`.`default`.`retail`;")
retail_df.head()

Row(invoice_no='561807', stock_code='23286', description='BLUE VINTAGE SPOT BEAKER', quantity=8, invoice_date=datetime.datetime(2011, 7, 29, 15, 2), unit_price=0.85, customer_id=17491.0, country='United Kingdom')

In [0]:
%python
retail_df.printSchema()
retail_df.toPandas().describe()

root
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- country: string (nullable = true)



,quantity,unit_price,customer_id
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


## Total Invoice Amount Distribution

In [0]:
%python
from pyspark.sql.functions import col, sum, max, min, median, mode, mean

df_filtered = retail_df.filter((col("quantity") > 0) & (col("unit_price") > 0))

df_filtered = df_filtered.withColumn("amount", col("quantity") * col("unit_price"))
invoice_totals = df_filtered.groupBy("invoice_no").agg(sum("amount").alias("amount"))
print(f"Max: ${invoice_totals.select(max("amount")).first()[0]}")
print(f"Min: ${invoice_totals.select(min("amount")).first()[0]}")
print(f"Mean: ${invoice_totals.select(mean("amount")).first()[0]}")
print(f"Median: ${invoice_totals.select(median("amount")).first()[0]}")
print(f"Mode: ${invoice_totals.select(mode("amount")).first()[0]}")
display(invoice_totals)

Max: $168469.6
Min: $0.19
Mean: $523.3037611158238
Median: $304.3150000000001
Mode: $15.0


invoice_no,amount
489677,192.0
491045,303.2
491658,155.05999999999997
493542,118.75
493977,275.95
494244,6711.0
494277,1335.92
495185,2507.06
495783,48.96
496171,199.29999999999998


Databricks visualization. Run in Databricks to view.

In [0]:
%python
q85 = invoice_totals.approxQuantile("amount", [0.85], 0.01)[0]
q85_invoices = invoice_totals.filter(col("amount") <= q85)

print(f"Max: ${q85_invoices.select(max("amount")).first()[0]}")
print(f"Min: ${q85_invoices.select(min("amount")).first()[0]}")
print(f"Mean: ${q85_invoices.select(mean("amount")).first()[0]}")
print(f"Median: ${q85_invoices.select(median("amount")).first()[0]}")
print(f"Mode: ${q85_invoices.select(mode("amount")).first()[0]}")
display(q85_invoices)

Max: $692.96
Min: $0.19
Mean: $266.9204015252676
Median: $253.02
Mode: $15.0


invoice_no,amount
489677,192.0
491045,303.2
491658,155.05999999999997
493542,118.75
493977,275.95
495783,48.96
496171,199.29999999999998
496233,188.82999999999998
496427,291.14000000000004
497229,312.5899999999999


Databricks visualization. Run in Databricks to view.

## Monthly Placed and Canceled Orders

In [0]:
%python
from pyspark.sql.functions import sum, when, date_format

retail_df = retail_df.withColumn(
    "amount", col("quantity") * col("unit_price")
)

invoice_totals = (
    retail_df
    .groupBy("invoice_no", "invoice_date")
    .agg(sum("amount").alias("total_amount"))
)

invoice_totals = invoice_totals.withColumn(
    "is_cancelled",
    col("total_amount") < 0
)

invoice_totals = invoice_totals.withColumn(
    "InvoiceYearMonth",
    date_format(col("invoice_date"), "yyyy-MM")
)

summary = (
    invoice_totals
    .groupBy("InvoiceYearMonth")
    .agg(
        sum(when(~col("is_cancelled"), 1).otherwise(0)).alias("Placement"),
        sum(when(col("is_cancelled"), 1).otherwise(0)).alias("Cancellation")
    )
)

summary = summary.withColumn(
    "Placement",
    col("Placement") - col("Cancellation")
)

In [0]:
%python
display(summary)

InvoiceYearMonth,Placement,Cancellation
2010-08,1340,273
2010-11,2524,576
2010-02,1491,239
2010-04,1284,305
2011-05,1539,314
2010-03,1558,407
2010-01,1034,300
2010-09,1642,371
2010-06,1504,357
2011-03,1352,319


Databricks visualization. Run in Databricks to view.

## Monthly Sales

In [0]:
%python
retail_df = retail_df.withColumn(
    "InvoiceYearMonth",
    date_format(col("invoice_date"), "yyyy-MM")
)

monthly_sales = (
    retail_df
    .groupBy("InvoiceYearMonth")
    .agg(sum("amount").alias("amount"))
)

display(monthly_sales)

InvoiceYearMonth,amount
2010-02,533091.4260000042
2010-04,590580.4319999823
2010-03,765848.7609999765
2010-01,624032.8919999955
2010-06,679786.6099999842
2010-07,575236.3600000103
2009-12,799847.1100000143
2010-05,615322.8300000005
2011-05,723333.5100000103
2011-03,683267.0800000189


Databricks visualization. Run in Databricks to view.

## Monthly Sales Growth

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.orderBy("InvoiceYearMonth")

monthly_growth = monthly_sales.withColumn(
    "pct_change",
    (col("amount") - lag("amount").over(Window.orderBy("InvoiceYearMonth"))) /
    lag("amount").over(Window.orderBy("InvoiceYearMonth"))
)

display(monthly_growth)

InvoiceYearMonth,amount,pct_change
2009-12,799847.1100000143,null
2010-01,624032.8919999955,-0.21980978089677128
2010-02,533091.4260000042,-0.14573184709627765
2010-03,765848.7609999765,0.4366180426994347
2010-04,590580.4319999823,-0.22885501410375667
2010-05,615322.8300000005,0.0418950521544184
2010-06,679786.6099999842,0.10476416095268813
2010-07,575236.3600000103,-0.153798631014483
2010-08,656776.3399999854,0.1417503928297815
2010-09,853650.4309999745,0.29975819622246663


Databricks visualization. Run in Databricks to view.

## Monthly Active Users

In [0]:
%python
from pyspark.sql.functions import countDistinct

monthly_active_users = (retail_df.groupBy("InvoiceYearMonth").agg(countDistinct("customer_id").alias("active_users")))

display(monthly_active_users)

InvoiceYearMonth,active_users
2010-08,964
2010-11,1683
2010-02,807
2010-04,998
2011-05,1079
2010-03,1111
2010-01,786
2010-09,1202
2010-06,1095
2011-03,1020


Databricks visualization. Run in Databricks to view.

## New and Existing Users

In [0]:
%python

first_purchase = (
  retail_df.groupby("customer_id").agg(min("InvoiceYearMonth").alias("FirstPurchaseMonth"))
)

retail_df_copy = retail_df.join(first_purchase, on="customer_id", how="left")

retail_df_copy = retail_df_copy.withColumn("NewUser", col("InvoiceYearMonth") == col("FirstPurchaseMonth"))

monthly_users = (
  retail_df_copy
  .dropDuplicates(["customer_id", "InvoiceYearMonth"])
  .groupby("InvoiceYearMonth")
  .pivot("NewUser")
  .agg(countDistinct("customer_id"))
  .fillna(0)
)

monthly_users = monthly_users.selectExpr(
    "InvoiceYearMonth",
    "`true` as NewUserCount",
    "`false` as ExUserCount"
)

In [0]:
%python
display(monthly_users)

InvoiceYearMonth,NewUserCount,ExUserCount
2010-08,158,806
2010-11,322,1361
2010-02,363,444
2010-04,291,707
2011-05,108,971
2010-03,436,675
2010-01,394,392
2010-09,242,960
2010-06,269,826
2011-03,178,842


Databricks visualization. Run in Databricks to view.

## RFM Segmentation

### Data Preparation

In [0]:
%python
from pyspark.sql.functions import lit, to_date

today = to_date(lit("2011-12-31"))

retail_df = retail_df.filter((col("quantity") > 0) & (col("unit_price") > 0))
retail_df = retail_df.na.drop()
retail_df.toPandas().describe([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
quantity,805549.0,13.290522,143.634088,1.000,1.00,1.00,1.00,2.00,5.00,12.00,24.00,36.0,128.00,80995.0
unit_price,805549.0,3.206561,29.199173,0.001,0.29,0.42,0.55,1.25,1.95,3.75,6.75,8.5,14.95,10953.5
customer_id,805549.0,15331.954970,1696.737039,12346.000,12422.00,12681.00,12979.00,13982.00,15271.00,16805.00,17716.00,17913.0,18204.00,18287.0
amount,805549.0,22.026505,224.041928,0.001,0.55,1.25,2.08,4.95,11.85,19.50,35.40,67.5,201.60,168469.6


### Finding RFM Score
RFM consists of Recency, Frequency, Monetary initials of expressions.

It is a technique that helps determine marketing and sales strategies based on buying habits of customers.

Recency: Time since customer last purchase

Frequency: Total number of purchases.

Monetary: Total spending by the customer.

In [0]:
%python
from pyspark.sql.functions import datediff
# finding Recency and Monetary values.
df_x = (
    retail_df
    .groupBy("customer_id")
    .agg(
        sum("amount").alias("amount_x"),
        datediff(today, max("invoice_date")).alias("invoice_date")
    )
)  # recency value
# x.max()).days; last shopping date of customers

In [0]:
%python
from pyspark.sql.functions import count
df_y = retail_df.groupby("customer_id", "invoice_no").agg(sum("amount").alias("amount"))
df_z = df_y.groupby("customer_id").agg(count("amount").alias("amount_y"))
# finding the frequency value per capita

In [0]:
%python
# creating the RFM table
rfm_table = df_x.join(df_z, on="customer_id", how="inner")

In [0]:
%python
rfm_table = (
  rfm_table
    .withColumnRenamed("invoice_date", "Recency")
    .withColumnRenamed("amount_x", "Monetary")
    .withColumnRenamed("amount_y", "Frequency")
)

In [0]:
%python
from pyspark.sql.functions import ntile, concat_ws

recency_window = Window.orderBy(col("Recency").desc())
freq_window = Window.orderBy(col("Frequency").asc())
monetary_window = Window.orderBy(col("Monetary").asc())

rfm_table = (
            rfm_table
             .withColumn("RecencyScore", ntile(5).over(recency_window))
             .withColumn("FrequencyScore", ntile(5).over(freq_window))
             .withColumn("MonetaryScore", ntile(5).over(monetary_window))
             )

rfm_table = rfm_table.withColumn(
    "RFM_SCORE",
    concat_ws("", col("RecencyScore"), col("FrequencyScore"), col("MonetaryScore"))
)

In [0]:
%python
rfm_table = rfm_table.withColumn(
    "Segment",
    concat_ws("", col("RecencyScore"), col("FrequencyScore"))
)

from pyspark.sql.functions import when

rfm_table = rfm_table.withColumn(
    "Segment",
    when(col("Segment").rlike(r"[1-2][1-2]"), "Hibernating")
    .when(col("Segment").rlike(r"[1-2][3-4]"), "At Risk")
    .when(col("Segment").rlike(r"[1-2]5"), "Can't Lose")
    .when(col("Segment").rlike(r"3[1-2]"), "About to Sleep")
    .when(col("Segment").rlike(r"33"), "Need Attention")
    .when(col("Segment").rlike(r"[3-4][4-5]"), "Loyal Customers")
    .when(col("Segment").rlike(r"41"), "Promising")
    .when(col("Segment").rlike(r"51"), "New Customers")
    .when(col("Segment").rlike(r"[4-5][2-3]"), "Potential Loyalists")
    .when(col("Segment").rlike(r"5[4-5]"), "Champions")
    .otherwise("Other")
)

In [0]:
%python
summary = (
    rfm_table
    .groupBy("Segment")
    .agg(
        mean("Recency").alias("Recency_mean"),
        mean("Frequency").alias("Frequency_mean"),
        mean("Monetary").alias("Monetary_mean"),
        count("Recency").alias("Recency_count"),
        count("Frequency").alias("Frequency_count"),
        count("Monetary").alias("Monetary_count")
    )
)
display(summary)

Segment,Recency_mean,Frequency_mean,Monetary_mean,Recency_count,Frequency_count,Monetary_count
Hibernating,477.1611922141119,1.3065693430656935,449.9006763990271,1644,1644,1644
About to Sleep,129.04534606205252,1.4128878281622912,538.3337303102629,419,419,419
Potential Loyalists,48.48754914809961,2.0288335517693317,932.6087811271293,763,763,763
At Risk,385.5357142857143,4.298136645962733,1535.6636801242223,644,644,644
Need Attention,133.9892086330935,3.449640287769784,1359.2972661870494,278,278,278
Loyal Customers,84.56308213378493,9.831498729889924,4200.169547840811,1181,1181,1181
Champions,30.023728813559323,18.381920903954803,10355.231079096044,885,885,885
Can't Lose,351.84375,16.796875,8984.66159375,64,64,64


### Remark:
3 segments selected for evaluation are "Can't Lose", "Hibernating" and "Champions".

Number of customers for segments:

Can't Lose = 64, Hibernating = 1644, Champions = 885

Can't Lose Segment;

The last shopping date of the customers is on average 352 days before.
Customers have made an average of 17 purchases.
Customers spent an average of £ 8995.
Hibernating Segment;

The last shopping date of the customers is 477 days before average.
Customers made an average of 1 purchases.
Customers spent an average of £ 450.
Champions Segment;

The last shopping date of the customers is 30 days before average.
Customers made an average of 18 purchases.
Customers spent an average of £ 10355.
Can't Lose Segment;

Customers in this segment have not recently made a purchase. For this reason, we need to prepare a discount and gift campaign for this segment. These customers made a large number of purchases when they made purchases before. However, recency values are lower than they should be. The campaign to be implemented for these customers should include both items purchased and recommendations based on previous activities. New and popular products associated with the products that they were interested in can also be included in this campaign. Situations that will cause these customers to stop buying need to be investigated.
Hibernating Segment;

Customers in this segment have not made a purchase for a long time. However, by offering discounts, they may be attracted to another purchase.
Champions Segment;

Customers in this segment are responsible for most of the revenue. Campaigns should be implemented to ensure the continuity of the shopping of these customers.